# Session 2 Homework · Solutions (teacher copy)

Complete, idiomatic solutions with teaching notes. All code runs top-to-bottom against the inline dataset.

**Grading toward the success criterion** (concrete artifact, not a metric — named metrics begin Session 8):

- all five exercises attempted
- Ex 2: missing values shown as both count and percentage; `age_months` named as worst by percentage
- Ex 3: `phones_filled.isna().sum()` is `0` for every column
- Ex 4: ≥ 3 sentences, with a reason that depends on the dataset being small
- Ex 5: identifies the *buyer* as the harmed party and explains the false "perfect battery" claim

Accept reasoning that differs from these notes as long as it is honest and defended. Flag any answer that treats a missing value as a zero or a known-good value — that is the misconception this session targets.

## Exercise 1 · Load and look

In [ ]:
import pandas as pd
import numpy as np

phones = pd.DataFrame({
    "listing":            ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"],
    "brand":              ["Pixel", "Galaxy", "iPhone", "Redmi", "iPhone", "Galaxy", "Pixel", "Redmi", "iPhone", "Galaxy"],
    "price_thousands":    [22.0, 15.0, 38.0, 9.0, 41.0, 18.0, 25.0, 11.0, 35.0, 16.0],
    "storage_gb":         [128, 128, 256, 64, 256, 128, 128, 64, 256, 128],
    "battery_health_pct": [88.0, np.nan, 95.0, 70.0, np.nan, 82.0, 91.0, np.nan, 93.0, 79.0],
    "age_months":         [14.0, 26.0, np.nan, 33.0, 8.0, np.nan, 12.0, 40.0, 10.0, 22.0],
})
phones

In [ ]:
print("rows, columns:", phones.shape)
phones.info()

**Expected answers:**

1. 10 rows, 6 columns.
2. Numeric: `price_thousands`, `storage_gb`, `battery_health_pct`, `age_months`. Text: `brand` (and `listing` is an identifier, not a feature). Note for the student: `battery_health_pct` and `age_months` show fewer than 10 non-null values — those are the columns with holes.

## Exercise 2 · Detect the missing values

In [ ]:
print("missing count per column:")
print(phones.isna().sum())

In [ ]:
missing_percent = phones.isna().sum() / len(phones) * 100
print("missing percent per column:")
print(missing_percent.round(1))

**Expected answer:** `battery_health_pct` has 3 missing (30%) and `age_months` has 2 missing (20%). The worst-affected column is **`battery_health_pct`** at 30%.

*Teaching note:* 30% missing in a 10-row set is a lot — a good cue for the Ex 4 discussion that dropping here would be reckless.

## Exercise 3 · Handle them two ways

In [ ]:
phones_dropped = phones.dropna()
print("rows after dropping:", phones_dropped.shape[0], "of", phones.shape[0])

In [ ]:
phones_filled = phones.copy()

battery_median = phones_filled["battery_health_pct"].median()
phones_filled["battery_health_pct"] = phones_filled["battery_health_pct"].fillna(battery_median)

age_median = phones_filled["age_months"].median()
phones_filled["age_months"] = phones_filled["age_months"].fillna(age_median)

print(phones_filled.isna().sum())

**Expected answer:** dropping leaves **5 of 10** rows. The five holes fall across five distinct listings — B, E, H (missing battery health) and C, F (missing age) — so `dropna()` removes exactly those five. Losing half the data is clearly too costly. The filled version reports `0` missing everywhere.

*Acceptable alternative:* filling with the **mean** instead of the median is fine to accept, but ask the student which is safer if one phone were extremely old or nearly dead — the median resists those extremes.

## Exercise 4 · Decide, and justify

**Model answer (accept any well-reasoned variant of this):**

I would fill, not drop. This dataset has only 10 listings, and dropping every row with a missing value would throw away 5 of them — half the data. Each of those rows still has a real price, brand, and storage size that are useful, so deleting them to get rid of one unknown number is a bad trade. With so little data, every row is precious, so I fill the two numeric holes with their medians and keep all 10 listings.

*Grading:* require at least three sentences and a reason that explicitly depends on the dataset being small / each row being valuable. A student who argues for dropping should only pass if they acknowledge the 50% loss and justify it anyway (hard to do honestly here — that tension is the point).

## Exercise 5 · Garbage in, garbage out

In [ ]:
careless = phones.copy()
careless["battery_health_pct"] = careless["battery_health_pct"].fillna(100)

print("phones advertised at 100% battery before the careless fill:", (phones["battery_health_pct"] == 100).sum())
print("phones advertised at 100% battery after the careless fill: ", (careless["battery_health_pct"] == 100).sum())
print()
print("honest median battery health (holes ignored):", round(phones["battery_health_pct"].median(), 1), "%")
print("mean battery health after filling with 100:  ", round(careless["battery_health_pct"].mean(), 1), "%")

**Expected reflection:**

1. It misleads the **buyer**. Three phones with unknown battery health are now advertised as having perfect (100%) batteries. A buyer trusting that number could pay too much, or buy a phone whose battery is actually worn out — a real cost to a real person.
2. A missing value means *we don't know*; a good value means *we checked and it's fine*. Treating "unknown" as "perfect" invents a fact, and any decision built on it is built on a lie — garbage in, garbage out.

*Teaching note:* tie this back to the classwork's age-filled-with-0 demonstration. Same mistake, different column: a careless fill is wrong information, not missing information.